In [ ]:
# scarches
import scanpy as sc
import seaborn as sns
import torch
import scarches as sca
from scarches.dataset.trvae.data_handling import remove_sparsity
import matplotlib.pyplot as plt
import numpy as np
import gdown
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from mousipy import translate
sc.settings.set_figure_params(dpi=200, frameon=False)
sc.set_figure_params(dpi=200)
sc.set_figure_params(figsize=(4, 4))
torch.set_printoptions(precision=3, sci_mode=False, edgeitems=7)

# set anndata keys
condition_key = 'donor_id'
cell_type_key = 'main_class'

# read adata file
source_adata = sc.read_h5ad('read_single_nuc_reference_file.h5ad'
)
target_adata = sc.read_h5ad('input_combined_xenium_file.h5ad')
target_adata
# prepare genes for mapping
humanized_adata = target_adata
source_adata = source_adata.copy()
common_genes = np.intersect1d(humanized_adata.var_names, source_adata.var_names)
source_adata =source_adata[:, common_genes].copy()

#creating model for reference dataset
# Normalizing to median total counts
sc.pp.normalize_total(source_adata)
# Logarithmize the data
sc.pp.log1p(source_adata)
sc.pp.highly_variable_genes(
    source_adata, n_top_genes=2000, batch_key="donor_id", subset=True
)
# running SCVI for reference data
sca.models.SCVI.setup_anndata(source_adata, batch_key=condition_key, labels_key=cell_type_key)
vae = sca.models.SCVI(
    source_adata,
    n_layers=2,
    encode_covariates=True,
    deeply_inject_covariates=False,
    use_layer_norm="both",
    use_batch_norm="none",
)
vae.train(max_epochs = 100)
scanvae = sca.models.SCANVI.from_scvi_model(vae, unlabeled_category = "Unknown")
scanvae.train(max_epochs=100)
# create anndata file of latent representation
reference_latent = sc.AnnData(scanvae.get_latent_representation())
reference_latent.obs["cell_type"] = source_adata.obs[cell_type_key].tolist()
reference_latent.obs["batch"] = source_adata.obs[condition_key].tolist()
sc.pp.neighbors(reference_latent, n_neighbors=8)
sc.tl.leiden(reference_latent)
sc.tl.umap(reference_latent) 

sc.pl.umap(reference_latent,
           color=['batch', 'cell_type'],
           frameon=False,
           wspace=0.6
           )

#subset query data for common genes
humanized_adata.var_names_make_unique() 
humanized_adata = humanized_adata[:, source_adata.var_names].copy()
humanized_adata

#creating target data
sc.pp.neighbors(humanized_adata)
sc.tl.leiden(humanized_adata, flavor="igraph", n_iterations=2)

humanized_adata.obs.rename(columns={"sample": "donor_id"}, inplace=True)
humanized_adata.obs.rename(columns={"TMA_ID": "main_class"}, inplace=True)
reference_latent.obs['predictions'] = scanvae.predict()
print("Acc: {}".format(np.mean(reference_latent.obs.predictions == reference_latent.obs.cell_type)))

# save models 
ref_path = './ref'
scanvae.save(ref_path, overwrite=True)

humanized_adata.obs['orig_cell_types'] = humanized_adata.obs[cell_type_key].copy() 
humanized_adata.obs[cell_type_key] = scanvae.unlabeled_category_
# model 
model = sca.models.SCANVI.load_query_data(
    humanized_adata,
    ref_path,
    freeze_dropout = True,
)
model._unlabeled_indices = np.arange(humanized_adata.n_obs)
model._labeled_indices = []
print("Labelled Indices: ", len(model._labeled_indices))
print("Unlabelled Indices: ", len(model._unlabeled_indices))

#predicting
model.train(
    max_epochs=50,
    plan_kwargs=dict(weight_decay=0.0),
    check_val_every_n_epoch=10,
)

query_latent = sc.AnnData(model.get_latent_representation())
query_latent.obs['cell_type'] = humanized_adata.obs['orig_cell_types'].tolist()
query_latent.obs['batch'] = humanized_adata.obs[condition_key].tolist()

sc.pp.neighbors(query_latent)
sc.tl.leiden(query_latent)
sc.tl.umap(query_latent)
sc.pl.umap(
    query_latent,
    color=["batch", "cell_type"],
    frameon=False,
    wspace=0.6, save = 'mapping_umap.pdf'
)

#save model
surgery_path = './modelres'
model.save(surgery_path, overwrite=True)

query_latent.obs['predictions'] = model.predict()

df = query_latent.obs.groupby(["cell_type", "predictions"]).size().unstack(fill_value=0)
#save necessary files
df.to_csv('./prediction/resolution_prediction_df.csv')
norm_df.to_csv('./prediction/resolution_prediction_normalised_df.csv')
pd.DataFrame(common_genes).to_csv('./prediction/resolution_common_genes.csv')
pd.DataFrame(source_adata.var_names.tolist()).to_csv('./prediction/resolution__highly_variable_source.csv')
query_latent.write_h5ad('file_after_predictions.h5ad')
query_latent.obs.to_csv('predictions_incsv.csv')

# read file to plot mapping predictions with marker genes
df = pd.read_csv('./prediction/resolution_prediction_df.csv', index_col = 0)
norm_df = df / df.sum(axis=0)
row_sums = df.sum(axis=1)
percentage_df = df.divide(row_sums, axis=0) * 100
fig, ax = plt.subplots(figsize=(55, 45))
pastel_cmap = sns.color_palette("Blues", as_cmap=True)

heatmap = sns.heatmap(
    df,
    annot=False,
    cmap=pastel_cmap,
    cbar=True,
    linewidths=0,  
    square=True,   
    ax=ax
)
ax.grid(False)

for i in range(df.shape[0]):
    for j in range(df.shape[1]):
        raw_value = df.iloc[i, j]
        percentage_value = percentage_df.iloc[i, j]
        
        text_color = "black" if raw_value < df.to_numpy().max() * 0.6 else "white"
        
        plt.text(j + 0.5, i + 0.4, f"{raw_value}",
                 ha='center', va='center', fontsize=9,
                 fontweight="bold", color=text_color)
        
        plt.text(j + 0.5, i + 0.7, f"{percentage_value:.1f}%",
                 ha='center', va='center', fontsize=8,
                 fontstyle="italic", color="dimgray")

ax.set_xlabel("Predicted Cell Type", fontsize=12, fontweight='bold')
ax.set_ylabel("Actual Cell Type", fontsize=12, fontweight='bold')

for _, spine in ax.spines.items():
    spine.set_visible(True)
    spine.set_color('black')
    spine.set_linewidth(1.5)

ax.tick_params(axis='both', which='both', length=0)

plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)

plt.title("Heatmap of Cell Type Predictions", fontsize=14, fontweight="bold", pad=15)
plt.tight_layout() 
plt.savefig('heatmap_prediction_resolution.pdf')
plt.show()